# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import chromadb
import json
import time
from datetime import datetime

from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.rag import RAG
from typing import List, Optional, Dict, Any, Union, TypedDict
from lib.agents import Agent, AgentState, StateMachine, AgentStateGame, EntryPoint, Termination, Step
from lib.state_machine import Run
from lib.tooling import tool
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult


In [3]:
import os
from dotenv import load_dotenv, find_dotenv
from tavily import TavilyClient

load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Environment variables are loaded in the setup cell above.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
@tool
def retrieve_game_tool(query: str) -> list[dict[str, Any]]:
    """Find up to 10 game records relevant to a question.
        args:
            - query: a question about game industry. 

    """
    count = collection.count()
    if count == 0:
        return []

    results = collection.query(
        query_texts=[query],
        n_results=min(10, count),
        include=["metadatas"],
    )
    return (results["metadatas"] or [[]])[0]

In [6]:
def test_game_outputs(query: str, n_results: int = 5) -> None:
    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas"],
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    for i, (document, metadata) in enumerate(zip(documents, metadatas), start=1):
        print(f"Result {i}")
        print(f"Document: {document}")
        print(f"Metadata: {metadata}")
        print("-" * 40)

In [7]:
test_game_outputs("realistic racing games") 

Result 1
Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
Metadata: {'YearOfRelease': 1997, 'Name': 'Gran Turismo', 'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment', 'Platform': 'PlayStation 1', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'}
----------------------------------------
Result 2
Document: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
Metadata: {'Platform': 'PlayStation 3', 'Publisher': 'Sony Computer Entertainment', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', 'Genre': 'Racing', 'Name': 'Gran Turismo 5', 'YearOfRelease': 2010}
----------------------------------------
Result 3
Doc

#### Evaluate Retrieval Tool

In [8]:
retrieval_evaluator = AgentEvaluator()

@tool
def evaluate_retrieval_tool(
    question: str,
    documents: list[dict[str, Any]],
) -> dict[str, Any]:
    """ Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
        - question: original question from user
        - retrieved documents most similar to the user query in the Vector database
    The result includes:
        - useful: wether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    report = retrieval_evaluator.evaluate_retrieval(question, documents)
    return report.model_dump()

question = "find me some realistic car race games"
documents = retrieve_game_tool(question)

report = evaluate_retrieval_tool(question, documents)
print(report)

{'useful': True, 'description': "The retrieved documents contain several entries for realistic car racing games, specifically 'Gran Turismo' and 'Gran Turismo 5', which are well-known for their realistic driving physics and comprehensive racing simulations. These entries provide sufficient information about the games, including their platforms, publishers, genres, and descriptions, making them relevant to the user's request for realistic car race games."}


#### Game Web Search Tool

In [9]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.
@tool
def game_web_search(question:str):
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=question,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }
    
    return formatted_results

### Agent

In [10]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
tools=[retrieve_game_tool, evaluate_retrieval_tool, game_web_search]
agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions = """
        You are a video game research assistant.

        When the user states a preference, call register_memory to save it.
        When the user asks about their preferences, call search_memory before answering.
        If no relevant memory is found, say you don't know. Do not invent preferences.
        Do not use game research or web search for these preference-only requests.

        For game questions, first call retrieve_game_tool, then evaluate_retrieval_tool.
        If the results are useful, answer using them. Otherwise, call game_web_search
        and cite the source URLs.
    """
)

In [11]:
# TODO: Invoke your agent
# - When was Pokémon Gold and Silver released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X released for PlayStation 5?
session_id = "Udaplay"
if session_id in agent.memory.get_all_sessions():
    agent.reset_session(session_id)
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")
run1 = agent.invoke("When was Pokémon Gold and Silver released?", session_id)
run2 = agent.invoke("Which one was the first 3D platformer Mario game?", session_id)
run3 = agent.invoke("Was Mortal Kombat X released for PlayStation 5?", session_id)
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


### (Optional) Advanced — Stand Out

In [12]:
# TODO: Update your agent with long-term memory
"""run4 = agent.invoke(
    query="I prefer dark mode",
    session_id="session_1"
)"""
run5 = agent.invoke(
    query="What are my lighting preferences?",
    session_id="session_2"
)
runs = agent.get_session_runs(session_id) + agent.get_session_runs("session_2")

for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

# Run 1 {'run_id': 'dadd9a2b-0fcd-4cbc-a21b-9da5d65d37b2', 'start_timestamp': '2026-11-08 10:54:22.087351', 'end_timestamp': '2026-11-08 10:54:30.145122', 'snapshot_counts': 7}
Messages:
 -> (role = system, content = 
        You are a video game research assistant.

        When the user states a preference, call register_memory to save it.
        When the user asks about their preferences, call search_memory before answering.
        If no relevant memory is found, say you don't know. Do not invent preferences.
        Do not use game research or web search for these preference-only requests.

        For game questions, first call retrieve_game_tool, then evaluate_retrieval_tool.
        If the results are useful, answer using 

In [22]:
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

workflow = StateMachine(AgentStateGame)

# Conceptual step logic

def retrieve_step(state:AgentStateGame):
    documents = retrieve_game_tool(query=state["user_query"])
    return {"documents": documents}

def evaluate_step(state:AgentStateGame):
    report = evaluate_retrieval_tool(
        question=state["user_query"],
        documents=state["documents"],
    )
    return {"evaluation": report}

def web_search_step(state:AgentStateGame):
    result = game_web_search(question=state["user_query"])
    return {"web_result": result}

def output_step(state:AgentStateGame):
    llm = LLM(
        model="gpt-4o-mini",
        temperature=0.3,
    )

    report = state.get("evaluation")
    if report is None:
        raise ValueError("Cannot answer without an evaluation report")

    if report["useful"]:
        knowledge = state.get("documents")
        if knowledge is None:
            raise ValueError("Retrieved documents are missing")
    else:
        knowledge = state.get("web_result")
        if knowledge is None:
            raise ValueError("Web search result is missing")

    result= llm.invoke(
        [SystemMessage(
            content=f"You are a video game research assistant use only the information given to you" 
            f"Use only the following knowledge to answer, do not use your own knowledge: {knowledge}"
            f"If the retrieve inforamtion is not relevant "
            f"use the search web information provided instead and and cite the source URLs."
        ),
        UserMessage(
            content=state["user_query"]
        )]
    )
    return{"output": result.content}

# Create steps
entry = EntryPoint()
retrieve = Step("documents", retrieve_step)
evaluate = Step("evaluation", evaluate_step)
web_search = Step("Search the web", web_search_step)
output = Step("Output", output_step)
termination = Termination()
workflow.add_steps([entry, retrieve, evaluate,web_search, output, termination])
#router logic
def after_evaluation(state: AgentStateGame):
    report = state.get("evaluation")
    if report is None:
        raise ValueError("Evaluation step did not produce a report")

    if report["useful"]:
        return output  # Generate an answer before terminating
    return web_search

# Connect steps with a loop in increment
workflow.connect(entry, retrieve)
# First retrieve the documents, then evaluate them.
workflow.connect(retrieve, evaluate)
# Only after evaluation, decide whether to search the web.
workflow.connect(evaluate, [web_search, output], after_evaluation)
# If web search runs, it needs a path to the output too.
workflow.connect(web_search, output)
# Ouput to the end
workflow.connect(output, termination)
initial_state: AgentStateGame= {"user_query":"Was Mortal Kombat X released for PlayStation 5?"}
run_object = workflow.run(initial_state)
print(run_object.get_final_state())

[StateMachine] Starting: __entry__
[StateMachine] Executing step: documents
[StateMachine] Executing step: evaluation
[StateMachine] Executing step: Search the web
[StateMachine] Executing step: Output
[StateMachine] Terminating: __termination__
{'user_query': 'Was Mortal Kombat X released for PlayStation 5?', 'documents': [{'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'YearOfRelease': 2023, 'Platform': 'PlayStation 5', 'Name': "Marvel's Spider-Man 2", 'Publisher': 'Sony Interactive Entertainment', 'Genre': 'Action-adventure'}, {'Genre': 'Action-adventure', 'Name': "Marvel's Spider-Man", 'YearOfRelease': 2018, 'Platform': 'PlayStation 4', 'Publisher': 'Sony Interactive Entertainment', 'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.'}, {'Genre': 'First-person shooter', 'Platform': 'Xbox Series X|S', 'Description': "Th

In [23]:
print(run_object.get_final_state()["output"])

Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015. While it is playable on PS5, the sources do not explicitly state that it was officially released for the PlayStation 5. However, gameplay videos and the PlayStation website indicate that the game can be played on PS5, suggesting backward compatibility. Therefore, it is not definitively confirmed that it was released for PS5, but it is playable on the system. 

For more details, you can check the following sources: 
- [Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
- [PlayStation](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved)
